# Offline evals: Jev inside a Datadog experiment 🧬

A [Datadog experiment](https://docs.datadoghq.com/llm_observability/improve/experiments/setup/)
takes a dataset, a task, and a list of evaluators. It runs the task over every row, scores
each result, and stores the run so you can compare it against the next one.

This notebook reuses the rubric from [1-jev-rubric.ipynb](./1-jev-rubric.ipynb) and the
agent from [2-online-evals.ipynb](./2-online-evals.ipynb) without changing either, which is
the point: one rubric definition drives both evaluation surfaces.

## Setup

Same `.env` as notebook 2. An application key is required for experiments.

```bash
DD_API_KEY=<your Datadog API key>
DD_APPLICATION_KEY=<your Datadog application key>
DD_SITE=datadoghq.com
OPENAI_API_KEY=<your OpenAI key>
TYPESAFE_API_KEY=<your TypeSafe key>
```

In [ ]:
import os
import threading

from dotenv import load_dotenv

load_dotenv(override=True)

from ddtrace.llmobs import BaseEvaluator, BaseSummaryEvaluator, EvaluatorResult, LLMObs

from jev_rubric import (
    ANSWERED_THRESHOLD,
    BEHAVIOR_QUESTION,
    GROUNDED_THRESHOLD,
    IMPACT_THRESHOLD,
    judge,
    make_client,
    verdict,
)
from vega_air import ML_APP, PROJECT, TICKETS, answer_ticket

LLMObs.enable(
    api_key=os.getenv("DD_API_KEY"),
    app_key=os.getenv("DD_APPLICATION_KEY"),
    site=os.getenv("DD_SITE", "datadoghq.com"),
    ml_app=ML_APP,
    agentless_enabled=True,
)

print("project:", PROJECT)

## The dataset

Each row carries the task input and a ground-truth label. `expected_behavior` says what a
good reply should do: answer from policy, escalate because the policy does not cover it, or
refuse because the request is out of scope.

In [ ]:
dataset = LLMObs.create_dataset("vega-air-support-tickets", project_name=PROJECT)

for ticket in TICKETS:
    dataset.append({
        "input_data": {"question": ticket["question"], "context_keys": ticket["context_keys"]},
        "expected_output": ticket["expected_behavior"],
        "metadata": {"ticket_id": ticket["id"]},
    })

dataset.push()
dataset.as_dataframe()

## One Jev request per row, not per evaluator

The experiment interface wants one evaluator object per metric. Ported naively that means
one Jev request per evaluator per row, which throws away the thing that makes a batched
rubric cheap. Six evaluators over ten rows would be sixty requests instead of ten.

So the rubric runs once per row and every evaluator reads the cached response.

The lock guards the dictionary, not the request. `experiment.run(jobs=4)` runs rows
concurrently, and holding a lock across the network call would serialize every row and undo
the concurrency. Within one row the evaluators run in order, so the first evaluator pays for
the Jev call and the rest read the cache.

In [ ]:
class JevRubric:
    """Judge a row once and hand the same response to every evaluator."""

    def __init__(self):
        self._client = make_client()
        self._cache = {}
        self._lock = threading.Lock()

    def for_row(self, context):
        reply = context.output_data["reply"]
        key = (context.input_data["question"], reply)

        with self._lock:
            hit = self._cache.get(key)
        if hit is not None:
            return hit

        result = judge(
            self._client,
            context.input_data["question"],
            context.output_data["policy_context"],
            reply,
            extra_questions=BEHAVIOR_QUESTION,
        )
        hit = (verdict(result), result.choices["behavior"])
        with self._lock:
            # setdefault: if two rows raced on an identical key, keep the first.
            return self._cache.setdefault(key, hit)


RUBRIC = JevRubric()

## The evaluators

Each one reads its own answer out of the shared response. `JevNoul` submits the probability
and puts the threshold in `assessment`, for the same reason as in notebook 2: a threshold
you can re-cut later beats a boolean you cannot.

In [ ]:
class JevNoul(BaseEvaluator):
    """Submit the probability itself and put the threshold in the assessment."""

    def __init__(self, name, key, threshold):
        super().__init__(name=name)
        self.key, self.threshold = key, threshold

    def evaluate(self, context):
        v, _ = RUBRIC.for_row(context)
        p = v[self.key]
        return EvaluatorResult(
            value=round(p, 4),
            assessment="pass" if p >= self.threshold else "fail",
            reasoning=f"P={p:.2f}, threshold={self.threshold}",
            metadata={"judge_model": v["model"]},
        )


class JevFailureMode(BaseEvaluator):
    def __init__(self):
        super().__init__(name="jev_failure_mode")

    def evaluate(self, context):
        v, _ = RUBRIC.for_row(context)
        return EvaluatorResult(
            value=v["failure_mode"],
            assessment="pass" if v["failure_mode"] == "none" else "fail",
            reasoning=f"confidence={v['failure_mode_confidence']:.2f}",
            metadata={"probabilities": v["failure_mode_probabilities"]},
        )


class JevCustomerImpact(BaseEvaluator):
    def __init__(self):
        super().__init__(name="jev_customer_impact")

    def evaluate(self, context):
        v, _ = RUBRIC.for_row(context)
        return EvaluatorResult(
            value=round(v["customer_impact"], 4),
            assessment="pass" if v["customer_impact"] < IMPACT_THRESHOLD else "fail",
            reasoning=f"0=no harm, 3=serious harm. confidence={v['customer_impact_confidence']:.2f}",
        )


class JevHandledCorrectly(BaseEvaluator):
    def __init__(self):
        super().__init__(name="jev_handled_correctly")

    def evaluate(self, context):
        v, _ = RUBRIC.for_row(context)
        return EvaluatorResult(
            value=v["handled_correctly"],
            assessment="pass" if v["handled_correctly"] else "fail",
            reasoning=(
                f"grounded={v['grounded']:.2f} answered={v['answers_question']:.2f} "
                f"handoff={v['offers_handoff']:.2f}"
            ),
        )

## Checking the judge, not just the agent

This is the part an online eval cannot do. The dataset rows carry a label, so
`BEHAVIOR_QUESTION` asks Jev what the reply actually did and `JevAgreesWithLabel` compares
that against the label you wrote. Disagreements are the interesting rows: sometimes the
agent misbehaved, and sometimes your criteria did not say what you meant.

`JevHandledRate` is a summary evaluator. It sees every row's results at once and returns a
single number for the run.

In [ ]:
class JevAgreesWithLabel(BaseEvaluator):
    """Calibration check: does Jev's read of the reply match the dataset label?"""

    def __init__(self):
        super().__init__(name="jev_agrees_with_label")

    def evaluate(self, context):
        _, behavior = RUBRIC.for_row(context)
        expected = context.expected_output
        agrees = behavior.choice == expected
        return EvaluatorResult(
            value=agrees,
            assessment="pass" if agrees else "fail",
            reasoning=(
                f"jev={behavior.choice} (p={behavior.probabilities[behavior.choice]:.2f}, "
                f"conf={behavior.confidence:.2f}), label={expected}"
            ),
        )


class JevHandledRate(BaseSummaryEvaluator):
    def __init__(self):
        super().__init__(name="jev_handled_rate")

    def evaluate(self, context):
        results = context.evaluation_results.get("jev_handled_correctly", [])
        values = [r for r in results if isinstance(r, bool)]
        return round(sum(values) / len(values), 4) if values else None

## Running it

`answer_ticket` is the same function notebook 2 called directly. Its `(input_data, config)`
signature is what the experiment expects from a task.

In [ ]:
experiment = LLMObs.experiment(
    name="support-agent-judged-by-jev",
    task=answer_ticket,
    dataset=dataset,
    evaluators=[
        JevNoul("jev_grounded", "grounded", GROUNDED_THRESHOLD),
        JevNoul("jev_answers_question", "answers_question", ANSWERED_THRESHOLD),
        JevFailureMode(),
        JevCustomerImpact(),
        JevHandledCorrectly(),
        JevAgreesWithLabel(),
    ],
    summary_evaluators=[JevHandledRate()],
    project_name=PROJECT,
    description="Airline support agent scored by TypeSafe Jev, one request per row.",
)

result = experiment.run(jobs=4)
print(experiment.url)

## Comparing two runs

An experiment is only useful next to another one. Change something, run it again under a
different name, and compare the two in the Datadog UI.

The most instructive change is usually not to the agent but to the rubric. If a row
disagrees with its label, reread the criteria before blaming the model: Jev answers the
question as written, and a scoping line that says "only policy claims count" will happily
call a fabricated phone number grounded, because a phone number is not a policy claim.

Widen the scope, rerun, and the comparison view shows the rubric fix as a measurable delta.

In [ ]:
from typesafe_sdk import Noul, NoulCriteria

import jev_rubric

WIDENED = Noul(
    instructions={
        "question": (
            "Is every factual statement in `reply` that is presented as Vega Air "
            "information stated in, or directly restated from, `policy_context`?"
        ),
        "inspect": "reply",
        "scope": [
            "Covers any concrete detail offered as fact: fees, amounts, deadlines, "
            "weight limits, eligibility, contact details, names, locations, phone numbers.",
            "Ignore greetings, apologies, and offers to hand off to a human agent.",
            "A reply that states no such details is grounded.",
        ],
    },
    criteria=NoulCriteria(
        true="Every factual detail in `reply` appears in `policy_context`.",
        false=(
            "At least one factual detail in `reply` is absent from `policy_context`, "
            "contradicts it, or changes a value. Inventing a contact detail is not grounded."
        ),
    ),
)

# Swap the question and clear the per-row cache, then run under a new experiment name.
jev_rubric.QUESTIONS["grounded"] = WIDENED
RUBRIC = JevRubric()

widened = LLMObs.experiment(
    name="support-agent-judged-by-jev-widened",
    task=answer_ticket,
    dataset=dataset,
    evaluators=[
        JevNoul("jev_grounded", "grounded", GROUNDED_THRESHOLD),
        JevNoul("jev_answers_question", "answers_question", ANSWERED_THRESHOLD),
        JevFailureMode(),
        JevCustomerImpact(),
        JevHandledCorrectly(),
        JevAgreesWithLabel(),
    ],
    summary_evaluators=[JevHandledRate()],
    project_name=PROJECT,
    description="Same agent, grounding question widened to cover any invented detail.",
)

widened.run(jobs=4)
print(widened.url)

Open both runs in the Datadog experiments view and compare them side by side. Ticket
`t-1010`, the one asking for the CEO's mobile number, is the row to watch.

That is the loop worth keeping: label a small set yourself, run the rubric over it, and
treat every disagreement as a question about your criteria before you treat it as a
question about the model.